In [ ]:
# Cell 1 — Imports and configuration
# Run this cell first. Change experiment settings only in this cell.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 42
N_TRAIN = 500
N_VAL = 200
N_TEST = 1000
P = 10
NOISE_STD = 1.0
CONTAMINATION_RATIO = 0.10
CONTAMINATION_MAGNITUDE = 5.0
CONTAMINATION_TYPE = "one_sided_response"
ROBUST_THRESHOLD = 2.5
ROBUST_MAX_ITERATIONS = 50
ROBUST_TOLERANCE = 1e-8

# Benchmark A — which response-outlier structures damage least squares?
BENCHMARK_RATIOS = [0.00, 0.05, 0.10, 0.20, 0.30]
BENCHMARK_MAGNITUDES = [2.0, 5.0, 10.0]
BENCHMARK_TYPES = ["one_sided_response", "symmetric_response"]
# Benchmark B — OLS versus residual-based robust weighting.
ROBUST_BENCHMARK_CONDITIONS = [
    (0.10, 5.0, "one_sided_response"),
    (0.20, 10.0, "one_sided_response"),
    (0.20, 10.0, "symmetric_response"),
]
# Benchmark C — bring outliers closer until robust weighting loses its advantage.
FAILURE_DISTANCES = [10.0, 8.0, 6.0, 4.0, 3.0, 2.0, 1.5, 1.0]
FAILURE_RATIO = 0.10
# Benchmark D — two legitimate generating relationships.
TWO_CLUSTER_FRACTION_B = 0.50
TWO_CLUSTER_BETA_SHIFT = 2.0
BENCHMARK_REPEATS = 20

if CONTAMINATION_TYPE not in BENCHMARK_TYPES:
    raise ValueError("CONTAMINATION_TYPE must be one of BENCHMARK_TYPES.")
if not 0.0 <= CONTAMINATION_RATIO <= 1.0:
    raise ValueError("CONTAMINATION_RATIO must be between 0 and 1.")

pd.set_option("display.precision", 3)
plt.style.use("seaborn-v0_8-whitegrid")


In [ ]:
# Cell 2 — Generate true beta
# beta_true is known because this is a controlled synthetic experiment.

rng = np.random.default_rng(SEED)
beta_true = rng.normal(loc=0.0, scale=1.0, size=P)

print("beta_true:")
print(pd.Series(beta_true, index=[f"x{i + 1}" for i in range(P)], name="coefficient"))


In [ ]:
# Cell 3 — Generate clean train / validation / test sets
# Each split is independently generated. Only the training split is contaminated later.

def generate_clean_set(n_samples, beta, noise_std, generator):
    """Return X and y from y = X @ beta + Normal(0, noise_std)."""
    X = generator.normal(loc=0.0, scale=1.0, size=(n_samples, len(beta)))
    noise = generator.normal(loc=0.0, scale=noise_std, size=n_samples)
    y = X @ beta + noise
    return X, y


X_train_clean, y_train_clean = generate_clean_set(N_TRAIN, beta_true, NOISE_STD, rng)
X_val, y_val = generate_clean_set(N_VAL, beta_true, NOISE_STD, rng)
X_test, y_test = generate_clean_set(N_TEST, beta_true, NOISE_STD, rng)

print(f"X_train_clean: {X_train_clean.shape}; y_train_clean: {y_train_clean.shape}")
print(f"X_val: {X_val.shape}; y_val: {y_val.shape}")
print(f"X_test: {X_test.shape}; y_test: {y_test.shape}")


In [ ]:
# Cell 4 — Inject response contamination into training data only
# X_train_clean and y_train_clean are never modified in place.

def inject_contamination(X_clean, y_clean, contamination_type, ratio, magnitude, noise_std, generator):
    """Return observed data and masks.

    one_sided_response adds +magnitude * noise_std to selected targets.
    symmetric_response adds either sign with equal probability. Validation and
    test sets are generated elsewhere and remain clean.
    """
    X_observed = X_clean.copy()
    y_observed = y_clean.copy()
    response_mask = generator.random(len(y_clean)) < ratio
    feature_mask = np.zeros_like(X_clean, dtype=bool)  # keeps the sample-table interface explicit

    if contamination_type == "one_sided_response":
        signs = np.ones(response_mask.sum())
    elif contamination_type == "symmetric_response":
        signs = generator.choice([-1.0, 1.0], size=response_mask.sum())
    else:
        raise ValueError("contamination_type must be 'one_sided_response' or 'symmetric_response'.")
    y_observed[response_mask] += signs * magnitude * noise_std
    return X_observed, y_observed, response_mask, feature_mask


X_train_observed, y_train_observed, response_mask, feature_mask = inject_contamination(
    X_train_clean, y_train_clean, CONTAMINATION_TYPE, CONTAMINATION_RATIO,
    CONTAMINATION_MAGNITUDE, NOISE_STD, rng,
)

affected_rows = response_mask
print(f"contamination type: {CONTAMINATION_TYPE}")
print(f"affected training rows: {affected_rows.sum()} / {N_TRAIN}")
print("Validation and test data remain clean.")


In [ ]:
# Cell 5 — Show clean versus contaminated training samples

feature_names = [f"x{i + 1}" for i in range(P)]
sample_table = pd.DataFrame(X_train_observed, columns=feature_names)
sample_table.insert(0, "row_id", np.arange(N_TRAIN))
sample_table["y_clean"] = y_train_clean
sample_table["y_observed"] = y_train_observed
sample_table["response_contaminated"] = response_mask

clean_example_ids = np.flatnonzero(~affected_rows)[:5]
contaminated_example_ids = np.flatnonzero(affected_rows)[:5]
print("Clean training examples (observed target equals clean target):")
display(sample_table.iloc[clean_example_ids])
print("Contaminated training examples:")
display(sample_table.iloc[contaminated_example_ids])


In [ ]:
# Cell 6 — Train ordinary least squares and residual-based robust regression

def add_intercept(X):
    return np.column_stack([np.ones(len(X)), X])


def fit_ols(X, y):
    """Least squares with lstsq, stable for near-singular designs."""
    return np.linalg.lstsq(add_intercept(X), y, rcond=None)[0]


def robust_residual_scale(residuals):
    """Median-absolute-deviation scale with a safe lower bound."""
    scale = 1.4826 * np.median(np.abs(residuals - np.median(residuals)))
    return max(scale, np.finfo(float).eps)


def fit_robust_weighted_regression(X, y, threshold, max_iterations, tolerance):
    """Huber-style iteratively reweighted least squares and final row weights."""
    design = add_intercept(X)
    theta = np.linalg.lstsq(design, y, rcond=None)[0]
    weights = np.ones(len(y))
    for _ in range(max_iterations):
        residuals = y - design @ theta
        cutoff = threshold * robust_residual_scale(residuals)
        weights = np.minimum(1.0, cutoff / np.maximum(np.abs(residuals), np.finfo(float).eps))
        root_weights = np.sqrt(weights)
        updated = np.linalg.lstsq(design * root_weights[:, None], y * root_weights, rcond=None)[0]
        if np.linalg.norm(updated - theta) <= tolerance * (1.0 + np.linalg.norm(theta)):
            theta = updated
            break
        theta = updated
    return theta, weights


theta_hat = fit_ols(X_train_observed, y_train_observed)
robust_theta_hat, robust_weights = fit_robust_weighted_regression(
    X_train_observed, y_train_observed, ROBUST_THRESHOLD, ROBUST_MAX_ITERATIONS, ROBUST_TOLERANCE
)

intercept_hat = theta_hat[0]
beta_hat = theta_hat[1:]
y_train_pred = add_intercept(X_train_observed) @ theta_hat
print(f"OLS intercept = {intercept_hat:.3f}")
print(f"OLS observed-training MSE = {np.mean((y_train_observed - y_train_pred) ** 2):.3f}")
print(f"robust fit down-weighted {(robust_weights < 1.0).sum()} / {N_TRAIN} training rows")
display(pd.DataFrame({
    "beta_true": beta_true, "OLS_beta_hat": beta_hat, "robust_beta_hat": robust_theta_hat[1:],
    "OLS_difference": beta_hat - beta_true, "robust_difference": robust_theta_hat[1:] - beta_true,
}, index=feature_names))


In [ ]:
# Cell 7 — Evaluate both fits on clean validation and test data

def regression_metrics(X, y, theta, beta_reference):
    predictions = add_intercept(X) @ theta
    residuals = y - predictions
    mse = np.mean(residuals ** 2)
    parameter_error = np.linalg.norm(theta[1:] - beta_reference)
    return {
        "MSE": mse,
        "MAE": np.mean(np.abs(residuals)),
        "R2": 1.0 - np.sum(residuals ** 2) / np.sum((y - y.mean()) ** 2),
        "parameter_L2_error": parameter_error,
        "relative_parameter_error": parameter_error / np.linalg.norm(beta_reference),
    }

metrics = [
    regression_metrics(X_val, y_val, theta_hat, beta_true),
    regression_metrics(X_test, y_test, theta_hat, beta_true),
    regression_metrics(X_val, y_val, robust_theta_hat, beta_true),
    regression_metrics(X_test, y_test, robust_theta_hat, beta_true),
]
evaluation_table = pd.DataFrame(metrics, index=[
    "OLS: clean validation", "OLS: clean test", "robust: clean validation", "robust: clean test",
])
display(evaluation_table)


In [ ]:
# Cell 8 — Run the four benchmarks from the robust-regression research plan

def experiment_data(seed, ratio, magnitude, contamination_type):
    generator = np.random.default_rng(seed)
    beta = generator.normal(0.0, 1.0, size=P)
    X_train, y_train = generate_clean_set(N_TRAIN, beta, NOISE_STD, generator)
    X_val_local, y_val_local = generate_clean_set(N_VAL, beta, NOISE_STD, generator)
    X_test_local, y_test_local = generate_clean_set(N_TEST, beta, NOISE_STD, generator)
    X_observed, y_observed, _, _ = inject_contamination(
        X_train, y_train, contamination_type, ratio, magnitude, NOISE_STD, generator
    )
    return beta, X_observed, y_observed, X_val_local, y_val_local, X_test_local, y_test_local


def compare_fits(seed, ratio, magnitude, contamination_type):
    beta, X_obs, y_obs, X_val_local, y_val_local, X_test_local, y_test_local = experiment_data(
        seed, ratio, magnitude, contamination_type
    )
    ols = fit_ols(X_obs, y_obs)
    robust, _ = fit_robust_weighted_regression(X_obs, y_obs, ROBUST_THRESHOLD, ROBUST_MAX_ITERATIONS, ROBUST_TOLERANCE)
    records = []
    for method, theta in [("OLS", ols), ("robust", robust)]:
        test = regression_metrics(X_test_local, y_test_local, theta, beta)
        records.append({"seed": seed, "method": method, "clean_test_MSE": test["MSE"],
                        "parameter_L2_error": test["parameter_L2_error"],
                        "relative_parameter_error": test["relative_parameter_error"]})
    return records

# A: outlier structure. OLS only isolates the failure mode of squared-error regression.
benchmark_a_records = []
for type_index, contamination_type in enumerate(BENCHMARK_TYPES):
    for ratio in BENCHMARK_RATIOS:
        for magnitude in BENCHMARK_MAGNITUDES:
            for repeat in range(BENCHMARK_REPEATS):
                record = compare_fits(SEED + 100000 * type_index + 1000 * repeat + int(ratio * 100) + int(magnitude), ratio, magnitude, contamination_type)[0]
                record.update({"contamination_type": contamination_type, "contamination_ratio": ratio,
                               "outlier_distance_sigma": magnitude})
                benchmark_a_records.append(record)
benchmark_a_results = pd.DataFrame(benchmark_a_records)

# B: robust weighting under a few clearly contaminated conditions.
benchmark_b_records = []
for condition_index, (ratio, magnitude, contamination_type) in enumerate(ROBUST_BENCHMARK_CONDITIONS):
    for repeat in range(BENCHMARK_REPEATS):
        for record in compare_fits(SEED + 200000 + 1000 * condition_index + repeat, ratio, magnitude, contamination_type):
            record.update({"contamination_ratio": ratio, "outlier_distance_sigma": magnitude,
                           "contamination_type": contamination_type})
            benchmark_b_records.append(record)
benchmark_b_results = pd.DataFrame(benchmark_b_records)

# C: failure boundary as outliers approach the normal-noise scale.
benchmark_c_records = []
for distance in FAILURE_DISTANCES:
    for repeat in range(BENCHMARK_REPEATS):
        for record in compare_fits(SEED + 300000 + 1000 * int(distance * 10) + repeat, FAILURE_RATIO, distance, "one_sided_response"):
            record.update({"contamination_ratio": FAILURE_RATIO, "outlier_distance_sigma": distance})
            benchmark_c_records.append(record)
benchmark_c_results = pd.DataFrame(benchmark_c_records)


def generate_two_cluster_set(n_samples, beta_a, beta_b, fraction_b, generator):
    X = generator.normal(0.0, 1.0, size=(n_samples, len(beta_a)))
    cluster_b = generator.random(n_samples) < fraction_b
    beta_by_row = np.where(cluster_b[:, None], beta_b, beta_a)
    y = np.einsum("ij,ij->i", X, beta_by_row) + generator.normal(0.0, NOISE_STD, size=n_samples)
    return X, y, cluster_b

# D: both groups are valid; report each group separately instead of claiming one beta is truth.
benchmark_d_records = []
for repeat in range(BENCHMARK_REPEATS):
    generator = np.random.default_rng(SEED + 400000 + repeat)
    beta_a = generator.normal(0.0, 1.0, size=P)
    beta_b = beta_a + TWO_CLUSTER_BETA_SHIFT * np.ones(P)
    X_train, y_train, train_b = generate_two_cluster_set(N_TRAIN, beta_a, beta_b, TWO_CLUSTER_FRACTION_B, generator)
    X_test_local, y_test_local, test_b = generate_two_cluster_set(N_TEST, beta_a, beta_b, TWO_CLUSTER_FRACTION_B, generator)
    ols = fit_ols(X_train, y_train)
    robust, weights = fit_robust_weighted_regression(X_train, y_train, ROBUST_THRESHOLD, ROBUST_MAX_ITERATIONS, ROBUST_TOLERANCE)
    for method, theta in [("OLS", ols), ("robust", robust)]:
        predictions = add_intercept(X_test_local) @ theta
        benchmark_d_records.append({"seed": SEED + 400000 + repeat, "method": method,
            "test_MSE_cluster_A": np.mean((y_test_local[~test_b] - predictions[~test_b]) ** 2),
            "test_MSE_cluster_B": np.mean((y_test_local[test_b] - predictions[test_b]) ** 2),
            "mean_training_weight_A": 1.0 if method == "OLS" else weights[~train_b].mean(),
            "mean_training_weight_B": 1.0 if method == "OLS" else weights[train_b].mean()})
benchmark_d_results = pd.DataFrame(benchmark_d_records)
print(f"completed A={len(benchmark_a_results)}, B={len(benchmark_b_results)}, C={len(benchmark_c_results)}, D={len(benchmark_d_results)} runs")


In [ ]:
# Cell 9 — Display figures in this notebook and export PNG copies
from pathlib import Path

OUTPUT_DIR = Path("release/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def mean_std_table(frame, group_columns, metrics):
    return frame.groupby(group_columns, as_index=False).agg(**{
        f"mean_{metric}": (metric, "mean") for metric in metrics
    }, **{
        f"std_{metric}": (metric, "std") for metric in metrics
    })


benchmark_a_summary = mean_std_table(
    benchmark_a_results,
    ["contamination_type", "contamination_ratio", "outlier_distance_sigma"],
    ["clean_test_MSE", "parameter_L2_error"],
)
benchmark_b_summary = mean_std_table(
    benchmark_b_results,
    ["contamination_type", "contamination_ratio", "outlier_distance_sigma", "method"],
    ["clean_test_MSE", "parameter_L2_error"],
)
benchmark_c_summary = mean_std_table(
    benchmark_c_results,
    ["outlier_distance_sigma", "method"],
    ["clean_test_MSE", "parameter_L2_error"],
)
benchmark_d_summary = mean_std_table(
    benchmark_d_results,
    ["method"],
    ["test_MSE_cluster_A", "test_MSE_cluster_B", "mean_training_weight_A", "mean_training_weight_B"],
)

print("Benchmark A — outlier structure")
display(benchmark_a_summary)
print("Benchmark B — robust weighting")
display(benchmark_b_summary)
print("Benchmark C — failure boundary")
display(benchmark_c_summary)
print("Benchmark D — two legitimate clusters")
display(benchmark_d_summary)

# A: compare contamination structures and distances as the ratio increases.
fig_a, axes_a = plt.subplots(1, 2, figsize=(15, 5), sharex=True)
for (kind, distance), data in benchmark_a_summary.groupby(["contamination_type", "outlier_distance_sigma"]):
    data = data.sort_values("contamination_ratio")
    label = f"{kind.replace('_response', '')}, {distance:g}σ"
    axes_a[0].plot(data["contamination_ratio"], data["mean_clean_test_MSE"], marker="o", label=label)
    axes_a[1].plot(data["contamination_ratio"], data["mean_parameter_L2_error"], marker="o", label=label)
axes_a[0].set(title="Benchmark A: clean-test MSE", xlabel="contamination ratio", ylabel="mean clean-test MSE")
axes_a[1].set(title="Benchmark A: parameter error", xlabel="contamination ratio", ylabel="mean L2 parameter error")
for axis in axes_a:
    axis.legend(fontsize=8)
fig_a.tight_layout()
fig_a.savefig(OUTPUT_DIR / "benchmark_a_outlier_structure.png", dpi=200, bbox_inches="tight")
plt.show()

# B: paired bars make the OLS-versus-robust comparison explicit for each condition.
conditions = benchmark_b_summary[["contamination_type", "contamination_ratio", "outlier_distance_sigma"]].drop_duplicates().reset_index(drop=True)
conditions["label"] = conditions.apply(
    lambda row: f"{row['contamination_type'].replace('_response', '')}\n{row['contamination_ratio']:.0%}, {row['outlier_distance_sigma']:g}σ", axis=1
)
fig_b, axes_b = plt.subplots(1, 2, figsize=(15, 5))
positions = np.arange(len(conditions))
bar_width = 0.36
for offset, method in [(-bar_width / 2, "OLS"), (bar_width / 2, "robust")]:
    merged = conditions.merge(
        benchmark_b_summary[benchmark_b_summary["method"] == method],
        on=["contamination_type", "contamination_ratio", "outlier_distance_sigma"], how="left",
    )
    axes_b[0].bar(positions + offset, merged["mean_clean_test_MSE"], bar_width, label=method)
    axes_b[1].bar(positions + offset, merged["mean_parameter_L2_error"], bar_width, label=method)
for axis, title, ylabel in [
    (axes_b[0], "Benchmark B: robust weighting", "mean clean-test MSE"),
    (axes_b[1], "Benchmark B: parameter recovery", "mean L2 parameter error"),
]:
    axis.set(title=title, xlabel="contamination condition", ylabel=ylabel, xticks=positions, xticklabels=conditions["label"])
    axis.legend()
fig_b.tight_layout()
fig_b.savefig(OUTPUT_DIR / "benchmark_b_robust_weighting.png", dpi=200, bbox_inches="tight")
plt.show()

# C: convergence of the OLS and robust lines marks the practical failure boundary.
fig_c, axes_c = plt.subplots(1, 2, figsize=(15, 5))
for method, data in benchmark_c_summary.groupby("method"):
    data = data.sort_values("outlier_distance_sigma")
    axes_c[0].plot(data["outlier_distance_sigma"], data["mean_clean_test_MSE"], marker="o", label=method)
    axes_c[1].plot(data["outlier_distance_sigma"], data["mean_parameter_L2_error"], marker="o", label=method)
for axis, title, ylabel in [
    (axes_c[0], "Benchmark C: failure boundary", "mean clean-test MSE"),
    (axes_c[1], "Benchmark C: parameter recovery", "mean L2 parameter error"),
]:
    axis.set(title=title, xlabel="outlier distance (σ)", ylabel=ylabel)
    axis.invert_xaxis()
    axis.legend()
fig_c.tight_layout()
fig_c.savefig(OUTPUT_DIR / "benchmark_c_failure_boundary.png", dpi=200, bbox_inches="tight")
plt.show()

# D is retained as a compact visual counterexample: neither cluster is contamination.
fig_d, axes_d = plt.subplots(1, 2, figsize=(12, 5))
for index, metric in enumerate(["mean_test_MSE_cluster_A", "mean_test_MSE_cluster_B"]):
    axes_d[index].bar(benchmark_d_summary["method"], benchmark_d_summary[metric])
    axes_d[index].set(title=f"Benchmark D: {metric.replace('mean_', '').replace('_', ' ')}", ylabel="mean test MSE")
fig_d.tight_layout()
fig_d.savefig(OUTPUT_DIR / "benchmark_d_two_clusters.png", dpi=200, bbox_inches="tight")
plt.show()
